# SCE-Net + DROCC для one-class compatibility (Fashion)

Ноутбук переводит задачу из pair-classification (`y in {0,1}`) в **one-class classification / anomaly detection**:
- в `train_pairs` и `val_pairs` только **позитивные пары** (`good/good-like`),
- в `test_pairs` есть и `good`, и `bad`,
- на инференсе используем **скалярный anomaly score** и подбираем порог на валидации/калибровочном сете.

Ключевая идея: сохраняем архитектурную логику SCE-Net (condition masks + condition weight branch + pair-context embedding), CLIP-признаки и image cache, но меняем objective на DROCC.


## 1) Что именно меняется относительно pair-classification

1. Убираем бинарный классификатор `good/bad` как основную цель обучения.
2. Вводим one-class objective:
   - стадия warm-up: учим scoring-head давать высокий score на позитивных парах,
   - стадия adversarial DROCC: генерируем синтетические «near-boundary negatives» вокруг позитивов и отталкиваем их вниз.
3. На тесте считаем `p_normal = sigmoid(score)`, а `anomaly_score = 1 - p_normal`.
4. Порог выбираем по целевой метрике (F1 / balanced accuracy / precision@recall) на отложенном наборе с разметкой.


In [ ]:
import os
import math
import json
import random
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import AutoProcessor, AutoModel
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_recall_curve


## 2) Конфиг (сохраняем структуру и ключевые поля, меняем `hf_model_name`)


In [ ]:
@dataclass
class Config:
    seed: int = 42
    device: str = 'cuda' if torch.cuda.is_available() else 'cpu'

    # Пути
    train_pairs_path: str = 'data/train_pairs_positive.csv'
    val_pairs_path: str = 'data/val_pairs_positive.csv'
    test_pairs_path: str = 'data/test_pairs_labeled.csv'
    items_path: str = 'data/items_catalog.csv'
    image_root: str = 'data/images'

    # Encoder
    hf_model_name: str = 'patrickjohncyh/fashion-clip'
    clip_trainable: bool = False

    # SCE-Net
    embed_dim: int = 512
    condition_masks: int = 8
    weight_hidden: int = 512
    mask_l1_lambda: float = 1e-4
    feat_l2_lambda: float = 1e-6

    # DROCC
    epochs: int = 20
    warmup_epochs: int = 3
    batch_size: int = 64
    num_workers: int = 4
    lr: float = 1e-4
    weight_decay: float = 1e-4

    drocc_radius: float = 1.0
    drocc_gamma: float = 2.0
    drocc_steps: int = 5
    drocc_step_size: float = 0.1
    drocc_lambda: float = 1.0

    # инференс
    threshold_metric: str = 'f1'

cfg = Config()
print(asdict(cfg))


## 3) Детально: DROCC и как он ложится на SCE-Net пары


### Интуиция DROCC (Deep Robust One-Class Classification, 2020)

DROCC учит decision boundary вокруг **плотной области нормальных объектов**. Вместо случайных отрицательных примеров алгоритм строит **adversarial negatives** рядом с нормальными данными на сферической оболочке (`r` .. `gamma*r`) и заставляет модель отделять их как anomalous.

### В терминах нашей задачи compatibility

- Объектом one-class является **пара товаров** `(sku1, sku2)`.
- Позитивная пара проходит через SCE-Net и даёт pair-context representation `z`.
- Scoring-head `f(z)` выдаёт логит «normality».
- Для каждой позитивной пары ищем adversarial точку `z_adv` в окрестности `z`:
  - максимизируем loss «сделать её тоже normal»,
  - проецируем на annulus с радиусами `[r, gamma*r]`.
- Затем в основном шаге обучения:
  - `z` push up (normal),
  - `z_adv` push down (anomaly).

Это снижает эффект «перемудрённых негативов из эвристик», потому что negative signal появляется из геометрии нормального manifold, а не из ручной генерации плохих пар.


In [ ]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(cfg.seed)


## 4) Данные и RAM-cache изображений (переиспользуем базовый пайплайн)


In [ ]:
items_df = pd.read_csv(cfg.items_path)
train_df = pd.read_csv(cfg.train_pairs_path)  # только позитивы
val_df = pd.read_csv(cfg.val_pairs_path)      # только позитивы
test_df = pd.read_csv(cfg.test_pairs_path)    # good/bad для оценки и порога

# ожидаем те же поля путей, что в baseline pair-classification
# sku1_path / sku2_path — абсолютные или относительные пути внутри image_root

processor = AutoProcessor.from_pretrained(cfg.hf_model_name)

IMAGE_CACHE: Dict[str, Image.Image] = {}
def load_img_to_cache(p: str):
    p = str(p)
    if p not in IMAGE_CACHE:
        full = Path(cfg.image_root) / p
        IMAGE_CACHE[p] = Image.open(full).convert('RGB')

for col in ['sku1_path', 'sku2_path']:
    for p in pd.concat([train_df[col], val_df[col], test_df[col]], axis=0).astype(str).unique().tolist():
        load_img_to_cache(p)

pair_collate = make_pair_collate(processor)


In [ ]:
class PairImageDataset(Dataset):
    """Переиспользуем интерфейс из pair-classification ноутбука."""
    def __init__(self, pair_df: pd.DataFrame, image_cache: Dict[str, Image.Image], with_labels: bool = True):
        self.df = pair_df.reset_index(drop=True)
        self.image_cache = image_cache
        self.with_labels = with_labels

    def __len__(self):
        return len(self.df)

    def _load_img(self, p: str):
        p = str(p)
        if p not in self.image_cache:
            raise KeyError(f'Image path not found in cache: {p}')
        return self.image_cache[p]

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img1 = self._load_img(row['sku1_path'])
        img2 = self._load_img(row['sku2_path'])
        if self.with_labels:
            y = np.float32(row['y'])
            return img1, img2, y
        return img1, img2


def make_pair_collate(processor):
    """Переиспользуем тот же collate-паттерн из baseline."""
    def collate_fn(batch):
        if len(batch[0]) == 3:
            imgs1, imgs2, ys = zip(*batch)
            ys = torch.tensor(ys, dtype=torch.float32)
        else:
            imgs1, imgs2 = zip(*batch)
            ys = None

        px1 = processor(images=list(imgs1), return_tensors='pt')['pixel_values']
        px2 = processor(images=list(imgs2), return_tensors='pt')['pixel_values']
        return (px1, px2, ys) if ys is not None else (px1, px2)
    return collate_fn


## 5) SCE-Net encoder (condition masks + condition weight branch) + DROCC scoring head


In [ ]:
class SCENetOneClass(nn.Module):
    """Сохраняем SCE-Net и делаем финальный pair embedding через concat(e1,e2)."""
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.backbone = AutoModel.from_pretrained(cfg.hf_model_name)
        if not cfg.clip_trainable:
            for p in self.backbone.parameters():
                p.requires_grad = False

        d = cfg.embed_dim
        m = cfg.condition_masks
        self.condition_masks = nn.Parameter(torch.randn(m, d) * 0.02)
        self.weight_branch = nn.Sequential(
            nn.Linear(2 * d, cfg.weight_hidden), nn.ReLU(),
            nn.Linear(cfg.weight_hidden, m)
        )
        self.scorer = nn.Sequential(
            nn.Linear(2 * d, d), nn.ReLU(), nn.Linear(d, 1)
        )

    def encode_items(self, pixel_values):
        out = self.backbone.get_image_features(pixel_values=pixel_values)
        return F.normalize(out, p=2, dim=-1)

    def pair_embedding(self, v1, v2):
        w = torch.softmax(self.weight_branch(torch.cat([v1, v2], dim=-1)), dim=-1)
        masked_1 = v1.unsqueeze(1) * self.condition_masks.unsqueeze(0)
        masked_2 = v2.unsqueeze(1) * self.condition_masks.unsqueeze(0)
        e1 = torch.bmm(w.unsqueeze(1), masked_1).squeeze(1)
        e2 = torch.bmm(w.unsqueeze(1), masked_2).squeeze(1)
        z = torch.cat([F.normalize(e1, p=2, dim=-1), F.normalize(e2, p=2, dim=-1)], dim=-1)
        z = F.normalize(z, p=2, dim=-1)
        return z, e1, e2

    def forward(self, img1, img2):
        v1 = self.encode_items(img1)
        v2 = self.encode_items(img2)
        z, e1, e2 = self.pair_embedding(v1, v2)
        logit = self.scorer(z).squeeze(-1)
        return logit, z, e1, e2


## 6) DROCC loss и adversarial generation в embedding-space


In [ ]:
def drocc_generate_adv(z, model, steps=5, step_size=0.1, r=1.0, gamma=2.0):
    z_adv = z.detach() + 0.001 * torch.randn_like(z)
    z_adv.requires_grad_(True)

    for _ in range(steps):
        logit_adv = model.scorer(z_adv).squeeze(-1)
        # maximize normal-score for adversarial crafting
        adv_obj = F.binary_cross_entropy_with_logits(logit_adv, torch.ones_like(logit_adv))
        grad = torch.autograd.grad(adv_obj, z_adv, retain_graph=False, create_graph=False)[0]
        z_adv = z_adv.detach() + step_size * F.normalize(grad, dim=-1)

        delta = z_adv - z
        dist = delta.norm(p=2, dim=-1, keepdim=True).clamp_min(1e-8)
        dist_proj = dist.clamp(min=r, max=gamma * r)
        z_adv = (z + delta / dist * dist_proj).detach()
        z_adv.requires_grad_(True)

    return z_adv.detach()


def one_class_loss(model, z, cfg: Config):
    # positives
    logit_pos = model.scorer(z).squeeze(-1)
    pos_loss = F.binary_cross_entropy_with_logits(logit_pos, torch.ones_like(logit_pos))

    # drocc adversarial negatives
    z_adv = drocc_generate_adv(
        z, model,
        steps=cfg.drocc_steps,
        step_size=cfg.drocc_step_size,
        r=cfg.drocc_radius,
        gamma=cfg.drocc_gamma,
    )
    logit_adv = model.scorer(z_adv).squeeze(-1)
    neg_loss = F.binary_cross_entropy_with_logits(logit_adv, torch.zeros_like(logit_adv))

    l1 = model.condition_masks.abs().mean()
    l2 = z.pow(2).mean()
    total = pos_loss + cfg.drocc_lambda * neg_loss + cfg.mask_l1_lambda * l1 + cfg.feat_l2_lambda * l2
    return total, {'pos_loss': pos_loss.item(), 'neg_loss': neg_loss.item()}


In [ ]:
train_loader = DataLoader(PairImageDataset(train_df, IMAGE_CACHE, with_labels=False), batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers, collate_fn=pair_collate)
val_loader = DataLoader(PairImageDataset(val_df, IMAGE_CACHE, with_labels=False), batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers, collate_fn=pair_collate)
test_loader = DataLoader(PairImageDataset(test_df, IMAGE_CACHE, with_labels=True), batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers, collate_fn=pair_collate)

model = SCENetOneClass(cfg).to(cfg.device)
opt = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=cfg.lr, weight_decay=cfg.weight_decay)


## 7) Тренировка (warm-up + DROCC phase)


In [ ]:
for epoch in range(cfg.epochs):
    model.train()
    losses = []
    for batch in tqdm(train_loader, desc=f'train epoch {epoch}'):
        px1, px2 = batch[0].to(cfg.device), batch[1].to(cfg.device)
        opt.zero_grad()
        logit, z, e1, e2 = model(px1, px2)

        if epoch < cfg.warmup_epochs:
            loss = F.binary_cross_entropy_with_logits(logit, torch.ones_like(logit))
        else:
            loss, _ = one_class_loss(model, z, cfg)

        loss.backward()
        opt.step()
        losses.append(loss.item())
    print(f'epoch={epoch} train_loss={np.mean(losses):.4f}')


## 8) Оценка на test и подбор порога


In [ ]:
@torch.no_grad()
def predict_scores(loader):
    model.eval()
    ys, scores = [], []
    for batch in tqdm(loader, desc='infer'):
        if len(batch) == 3:
            px1, px2, y = batch
            ys.extend(y.numpy().tolist())
        else:
            px1, px2 = batch
        px1, px2 = px1.to(cfg.device), px2.to(cfg.device)
        logit, _, _, _ = model(px1, px2)
        p_normal = torch.sigmoid(logit).cpu().numpy()
        scores.extend((1.0 - p_normal).tolist())
    return np.array(ys), np.array(scores)

y_true, anom_score = predict_scores(test_loader)
y_anom = 1 - y_true
print('ROC-AUC(anomaly):', roc_auc_score(y_anom, anom_score))
print('PR-AUC(anomaly):', average_precision_score(y_anom, anom_score))

prec, rec, thr = precision_recall_curve(y_anom, anom_score)
f1 = 2 * prec * rec / (prec + rec + 1e-8)
best_i = np.nanargmax(f1)
best_thr = thr[max(0, best_i - 1)] if len(thr) > 0 else 0.5
print('best threshold:', float(best_thr), 'best F1:', float(np.nanmax(f1)))


## 9) Практические рекомендации именно для вашего кейса

1. **Не выбрасывайте полностью bad-пары**: оставьте их только для evaluation/threshold calibration, но не для основного train objective.
2. **Критично подобрать `drocc_radius`**:
   - слишком маленький -> слабая граница,
   - слишком большой -> модель начинает резать валидные «редкие» good-пары.
3. **Стабилизация**:
   - первые `2-5` эпох только positive BCE,
   - затем включать DROCC.
4. **Диагностика «белая футболка + джинсы = bad»**:
   - проверить распределение anomaly_score по базовым canonical-парам,
   - добавить отдельный мониторинг по категориям (`top+bottom`, `dress+shoes`).
5. **Калибровка**:
   - глобальный порог + (опционально) category-aware пороги.
6. **Hard positive mining**:
   - если есть мета-инфа про «сильные образы», можно reweight позитивы, чтобы manifold строился вокруг действительно качественных комбинаций.


## 10) Построчное объяснение DROCC-пайплайна (что делает каждая строка и зачем)

Ниже разбор именно ключевых участков one-class/DROCC части. Идея: вы можете идти сверху вниз и понимать назначение каждой операции.

### A. `drocc_generate_adv(...)`

- `def drocc_generate_adv(z, model, steps=5, step_size=0.1, r=1.0, gamma=2.0):`
  - объявляем функцию генерации adversarial-отрицательных точек вокруг нормальных эмбеддингов `z`.
  - `steps` — число PGD-шагов,
  - `step_size` — длина шага,
  - `r` и `gamma` задают внутренний/внешний радиус annulus-области.

- `z_adv = z.detach() + 0.001 * torch.randn_like(z)`
  - стартуем от `z` с очень маленьким шумом, чтобы избежать нулевого градиента и симметрий.
  - `detach()` разрывает связь с графом базового `z`, чтобы атака не меняла его напрямую.

- `z_adv.requires_grad_(True)`
  - включаем вычисление градиента по `z_adv`, так как будем оптимизировать её как переменную атаки.

- `for _ in range(steps):`
  - итеративный подъём по целевой функции атаки (PGD-подобно).

- `logit_adv = model.scorer(z_adv).squeeze(-1)`
  - считаем normality-logit для текущих adversarial-точек.

- `adv_obj = F.binary_cross_entropy_with_logits(logit_adv, torch.ones_like(logit_adv))`
  - целевая функция атаки: сделать `z_adv` похожей на normal (таргет=1),
  - затем мы двигаемся по градиенту так, чтобы найти «сложные» точки, которые модель склонна считать нормальными.

- `grad = torch.autograd.grad(adv_obj, z_adv, retain_graph=False, create_graph=False)[0]`
  - берём градиент целевой функции по `z_adv`.

- `z_adv = z_adv.detach() + step_size * F.normalize(grad, dim=-1)`
  - обновляем `z_adv` в направлении нормированного градиента,
  - нормализация стабилизирует масштаб шага между объектами.

- `delta = z_adv - z`
  - вектор смещения от исходной нормальной точки.

- `dist = delta.norm(p=2, dim=-1, keepdim=True).clamp_min(1e-8)`
  - считаем L2-дистанцию и защищаемся от деления на 0.

- `dist_proj = dist.clamp(min=r, max=gamma * r)`
  - проецируем расстояние в диапазон annulus `[r, gamma*r]`:
    - не слишком близко (иначе это почти normal),
    - не слишком далеко (иначе слишком лёгкие аномалии).

- `z_adv = (z + delta / dist * dist_proj).detach()`
  - геометрическая проекция текущей точки на допустимую оболочку.

- `z_adv.requires_grad_(True)`
  - снова включаем градиент на следующую итерацию.

- `return z_adv.detach()`
  - возвращаем финальные adversarial-точки как константы для внешнего шага оптимизации.

---

### B. `one_class_loss(...)`

- `def one_class_loss(model, z, cfg: Config):`
  - функция финального one-class loss для батча нормальных пар.

- `logit_pos = model.scorer(z).squeeze(-1)`
  - предсказание normality для реальных позитивных пар.

- `pos_loss = F.binary_cross_entropy_with_logits(logit_pos, torch.ones_like(logit_pos))`
  - принуждаем нормальные пары иметь высокий normal-score.

- `z_adv = drocc_generate_adv(...)`
  - строим near-boundary adversarial-примеры вокруг `z`.

- `logit_adv = model.scorer(z_adv).squeeze(-1)`
  - normality-score для adversarial-точек.

- `neg_loss = F.binary_cross_entropy_with_logits(logit_adv, torch.zeros_like(logit_adv))`
  - заставляем модель считать adversarial-точки аномалиями (target=0).

- `l1 = model.condition_masks.abs().mean()`
  - L1-регуляризация condition masks: стимулирует разреженность/дизентангл.

- `l2 = z.pow(2).mean()`
  - L2-регуляризация эмбеддингов: ограничивает взрыв нормы.

- `total = pos_loss + cfg.drocc_lambda * neg_loss + cfg.mask_l1_lambda * l1 + cfg.feat_l2_lambda * l2`
  - итоговый objective:
    - fit normal-data,
    - отталкивание boundary-negatives,
    - структурные регуляризации.

- `return total, {'pos_loss': ..., 'neg_loss': ...}`
  - возвращаем loss + диагностику компонентов для мониторинга.

---

### C. Train loop (warm-up + DROCC)

- `logit, z, e1, e2 = model(px1, px2)`
  - получаем normality-logit и pair-embedding.

- `if epoch < cfg.warmup_epochs:`
  - на первых эпохах отключаем DROCC,
  - это стабилизирует старт и даёт модели сначала научиться распознавать позитивы как normal.

- `loss = BCE(logit, ones)`
  - warm-up: чистое one-class positive обучение.

- `else: loss, _ = one_class_loss(...)`
  - после warm-up включаем adversarial DROCC-шаг.

- `loss.backward(); opt.step()`
  - стандартный шаг обучения по итоговому objective.

---

### D. Inference / thresholding

- `p_normal = sigmoid(logit)`
  - вероятность нормальности пары.

- `anomaly_score = 1 - p_normal`
  - чем больше score, тем более пара «аномальная/несовместимая».

- `y_anom = 1 - y_true`
  - переводим оригинальную разметку (`good=1`, `bad=0`) в anomaly-таргет (`bad=1`).

- `precision_recall_curve(...)` и `best F1`
  - подбираем рабочий порог для продакшена на размеченном наборе.

---

### E. Почему это решает вашу боль с «перемудрёнными негативами»

- В бинарной постановке качество сильно зависит от того, насколько корректно вы синтезировали `y=0`.
- В DROCC-постановке отрицательные примеры формируются из геометрии normal-manifold, а не из ручных эвристик.
- Поэтому модель реже «ломается» на базовых валидных сочетаниях (например, белая футболка + джинсы), если гиперпараметры `r/gamma/steps` настроены адекватно.



## 11) Максимально детальный line-by-line разбор: от батча до порога

Ниже — расширенный построчный разбор уже **сквозного** пайплайна DROCC: data -> model -> adv -> loss -> train -> infer.

### 11.1 Batch preparation

1. `pair_collate = make_pair_collate(processor)`
   - создаём функцию, которая в момент сборки батча применяет processor к спискам PIL-изображений.
2. `px1 = processor(images=list(imgs1), return_tensors='pt')['pixel_values']`
   - превращаем список первых изображений пары в тензор `[B, C, H, W]`.
3. `px2 = processor(images=list(imgs2), return_tensors='pt')['pixel_values']`
   - аналогично для второй позиции в паре.
4. `ys = torch.tensor(ys, dtype=torch.float32)`
   - лейбл приводим к float для BCE-лоссов.

### 11.2 Forward inside `SCENetOneClass`

1. `v1 = self.encode_items(img1)` / `v2 = self.encode_items(img2)`
   - CLIP/FashionCLIP-признаки для каждого товара.
2. `out = self.backbone.get_image_features(...)`
   - получаем глобальные визуальные векторы размерности `D`.
3. `F.normalize(out, p=2, dim=-1)`
   - приводим к единичной норме; стабилизирует обучение и метрику расстояний.
4. `w = softmax(weight_branch(concat(v1,v2)))`
   - веса условий `M`, зависящие от контекста конкретной пары.
5. `masked_1 = v1 * C`, `masked_2 = v2 * C`
   - каждое condition-mask подчеркивает свою семантическую подобласть.
6. `e1/e2 = weighted sum over masks`
   - собираем condition-aware представления каждого айтема.
7. `z = concat(norm(e1), norm(e2))`
   - финальный pair-вектор в стиле pair-classification пайплайна: конкатенация двух позиций.
8. `logit = scorer(z)`
   - скаляр normality logit для one-class цели.

### 11.3 Adversarial generation (DROCC)

1. `z_adv = z.detach() + eps * noise`
   - старт около нормальной точки, но без привязки к её графу.
2. `requires_grad_(True)`
   - теперь `z_adv` — оптимизируемая переменная внутренней атаки.
3. `adv_obj = BCEWithLogits(scorer(z_adv), ones)`
   - ищем точки, которые модель склонна считать нормальными.
4. `grad = autograd.grad(adv_obj, z_adv)`
   - направление максимизации этой «обманной» цели.
5. `z_adv += step_size * normalize(grad)`
   - градиентный шаг в embedding-space.
6. `delta = z_adv - z`, `dist = ||delta||`
   - вычисляем, куда ушли от исходной нормальной точки.
7. `dist_proj = clamp(dist, r, gamma*r)`
   - проецируем в annulus: не слишком близко и не слишком далеко.
8. `z_adv = z + delta/dist * dist_proj`
   - окончательная геометрическая коррекция после шага.

### 11.4 One-class objective

1. `pos_loss = BCEWithLogits(scorer(z), ones)`
   - реальные positive-пары должны быть normal.
2. `neg_loss = BCEWithLogits(scorer(z_adv), zeros)`
   - near-boundary adversarial точки должны быть anomaly.
3. `l1 = |condition_masks|.mean()`
   - регуляризация масок для разреженности.
4. `l2 = z.pow(2).mean()`
   - контроль масштаба пространства.
5. `total = pos + lambda_adv*neg + lambda1*l1 + lambda2*l2`
   - итоговый баланс всех частей.

### 11.5 Training schedule

1. `if epoch < warmup_epochs: loss = pos-only`
   - без adversarial стадии на старте для устойчивости.
2. `else: loss = one_class_loss(...)`
   - включается DROCC.
3. `loss.backward(); opt.step()`
   - обновляем параметры scorer/weight_branch/masks (и backbone, если разморожен).

### 11.6 Inference and threshold

1. `p_normal = sigmoid(logit)`
   - вероятность «нормальной» (сочетаемой) пары.
2. `anomaly_score = 1 - p_normal`
   - score несовместимости.
3. `y_anom = 1 - y_true`
   - преобразуем разметку под anomaly-метрики.
4. `precision_recall_curve -> best F1`
   - выбираем практический порог для принятия решения good/bad.

### 11.7 Что тюнить в первую очередь

1. `drocc_radius` — ширина внутренней запретной зоны вокруг нормальных точек.
2. `drocc_gamma` — насколько далеко допускаются adversarial-кандидаты.
3. `drocc_steps`, `drocc_step_size` — качество/жесткость генерации трудных негативов.
4. `warmup_epochs` — насколько стабильно модель «встает» перед adversarial-фазой.
5. `drocc_lambda` — относительная сила отталкивания boundary-negatives.

